In [0]:
import pandas as pd

In [0]:
spark.sql("use catalog proyecto_final_prueba")

DataFrame[]

In [0]:
catalog = spark.sql("select current_catalog()").first()[0]
schema = "gold"
table = "dim_tiempo"

In [0]:
spark.sql(f"create schema if not exists {catalog}.{schema}")

DataFrame[]

In [0]:
spark.sql(f"drop table if exists {catalog}.{schema}.{table}")

DataFrame[]

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.{table} (
  id_tiempo BIGINT,
  fecha_hora_local TIMESTAMP,
  fecha DATE,
  anio INT,
  mes INT,
  dia INT,
  hora INT,
  dia_semana STRING,
  es_fin_de_semana BOOLEAN
)
""")

DataFrame[]

In [0]:
df_silver = spark.table(f"{catalog}.silver.weather").toPandas()
df_silver


,fecha_hora_local,latitude,longitude,temperature,humidity,wind_speed,weather_code,weather_desc
0,2026-06-01 03:00:00,-12.056238,-77.06198,19.0,93,8.0,3,Nublado
1,2026-06-01 04:00:00,-12.056238,-77.06198,19.0,92,8.1,3,Nublado
2,2026-06-01 05:00:00,-12.056238,-77.06198,18.8,92,8.3,3,Nublado
3,2026-06-01 06:00:00,-12.056238,-77.06198,19.1,89,9.1,3,Nublado
4,2026-06-01 07:00:00,-12.056238,-77.06198,19.4,90,5.5,3,Nublado
...,...,...,...,...,...,...,...,...
5443,2026-03-24 06:00:00,-12.056238,-77.06198,21.3,83,1.4,3,Nublado
5444,2026-03-24 07:00:00,-12.056238,-77.06198,21.7,80,5.5,3,Nublado
5445,2026-03-24 08:00:00,-12.056238,-77.06198,22.3,80,6.0,3,Nublado
5446,2026-03-24 09:00:00,-12.056238,-77.06198,22.9,81,6.6,1,Nublado


In [0]:
dim = pd.DataFrame({'fecha_hora_local': df_silver['fecha_hora_local'].unique()})
dim

,fecha_hora_local
0,2026-06-01 03:00:00
1,2026-06-01 04:00:00
2,2026-06-01 05:00:00
3,2026-06-01 06:00:00
4,2026-06-01 07:00:00
...,...
5443,2026-03-24 06:00:00
5444,2026-03-24 07:00:00
5445,2026-03-24 08:00:00
5446,2026-03-24 09:00:00


In [0]:
dim = dim.sort_values('fecha_hora_local').reset_index(drop=True)
dim['id_tiempo'] =dim['fecha_hora_local'].dt.strftime('%Y%m%d%H').astype(int)
dim

,fecha_hora_local,id_tiempo
0,2025-12-31 19:00:00,2025123119
1,2025-12-31 20:00:00,2025123120
2,2025-12-31 21:00:00,2025123121
3,2025-12-31 22:00:00,2025123122
4,2025-12-31 23:00:00,2025123123
...,...,...
5443,2026-08-15 14:00:00,2026081514
5444,2026-08-15 15:00:00,2026081515
5445,2026-08-15 16:00:00,2026081516
5446,2026-08-15 17:00:00,2026081517


In [0]:
dim['fecha'] = dim['fecha_hora_local'].dt.date
dim['anio'] = dim['fecha_hora_local'].dt.year
dim['mes'] = dim['fecha_hora_local'].dt.month
dim['dia'] = dim['fecha_hora_local'].dt.day
dim['hora'] = dim['fecha_hora_local'].dt.hour
dim

,fecha_hora_local,id_tiempo,fecha,anio,mes,dia,hora
0,2025-12-31 19:00:00,2025123119,2025-12-31,2025,12,31,19
1,2025-12-31 20:00:00,2025123120,2025-12-31,2025,12,31,20
2,2025-12-31 21:00:00,2025123121,2025-12-31,2025,12,31,21
3,2025-12-31 22:00:00,2025123122,2025-12-31,2025,12,31,22
4,2025-12-31 23:00:00,2025123123,2025-12-31,2025,12,31,23
...,...,...,...,...,...,...,...
5443,2026-08-15 14:00:00,2026081514,2026-08-15,2026,8,15,14
5444,2026-08-15 15:00:00,2026081515,2026-08-15,2026,8,15,15
5445,2026-08-15 16:00:00,2026081516,2026-08-15,2026,8,15,16
5446,2026-08-15 17:00:00,2026081517,2026-08-15,2026,8,15,17


In [0]:
dias_espanol = {
    0: 'Lunes', 1: 'Martes', 2: 'Miércoles', 
    3: 'Jueves', 4: 'Viernes', 5: 'Sábado', 6: 'Domingo'
}
dim['dia_semana'] = dim['fecha_hora_local'].dt.dayofweek.map(dias_espanol)
dim

,fecha_hora_local,id_tiempo,fecha,anio,mes,dia,hora,dia_semana
0,2025-12-31 19:00:00,2025123119,2025-12-31,2025,12,31,19,Miércoles
1,2025-12-31 20:00:00,2025123120,2025-12-31,2025,12,31,20,Miércoles
2,2025-12-31 21:00:00,2025123121,2025-12-31,2025,12,31,21,Miércoles
3,2025-12-31 22:00:00,2025123122,2025-12-31,2025,12,31,22,Miércoles
4,2025-12-31 23:00:00,2025123123,2025-12-31,2025,12,31,23,Miércoles
...,...,...,...,...,...,...,...,...
5443,2026-08-15 14:00:00,2026081514,2026-08-15,2026,8,15,14,Sábado
5444,2026-08-15 15:00:00,2026081515,2026-08-15,2026,8,15,15,Sábado
5445,2026-08-15 16:00:00,2026081516,2026-08-15,2026,8,15,16,Sábado
5446,2026-08-15 17:00:00,2026081517,2026-08-15,2026,8,15,17,Sábado


In [0]:
dim['es_fin_de_semana'] = dim['fecha_hora_local'].dt.dayofweek >= 5
columnas_dim = [
    'id_tiempo', 'fecha_hora_local', 'fecha', 'anio', 
    'mes', 'dia', 'hora', 'dia_semana', 'es_fin_de_semana'
]
dim = dim[columnas_dim]
dim

,id_tiempo,fecha_hora_local,fecha,anio,mes,dia,hora,dia_semana,es_fin_de_semana
0,2025123119,2025-12-31 19:00:00,2025-12-31,2025,12,31,19,Miércoles,False
1,2025123120,2025-12-31 20:00:00,2025-12-31,2025,12,31,20,Miércoles,False
2,2025123121,2025-12-31 21:00:00,2025-12-31,2025,12,31,21,Miércoles,False
3,2025123122,2025-12-31 22:00:00,2025-12-31,2025,12,31,22,Miércoles,False
4,2025123123,2025-12-31 23:00:00,2025-12-31,2025,12,31,23,Miércoles,False
...,...,...,...,...,...,...,...,...,...
5443,2026081514,2026-08-15 14:00:00,2026-08-15,2026,8,15,14,Sábado,True
5444,2026081515,2026-08-15 15:00:00,2026-08-15,2026,8,15,15,Sábado,True
5445,2026081516,2026-08-15 16:00:00,2026-08-15,2026,8,15,16,Sábado,True
5446,2026081517,2026-08-15 17:00:00,2026-08-15,2026,8,15,17,Sábado,True


In [0]:
df_spark = spark.createDataFrame(dim)
df_spark.display()

id_tiempo,fecha_hora_local,fecha,anio,mes,dia,hora,dia_semana,es_fin_de_semana
2025123119,2025-12-31T19:00:00.000Z,2025-12-31,2025,12,31,19,Miércoles,false
2025123120,2025-12-31T20:00:00.000Z,2025-12-31,2025,12,31,20,Miércoles,false
2025123121,2025-12-31T21:00:00.000Z,2025-12-31,2025,12,31,21,Miércoles,false
2025123122,2025-12-31T22:00:00.000Z,2025-12-31,2025,12,31,22,Miércoles,false
2025123123,2025-12-31T23:00:00.000Z,2025-12-31,2025,12,31,23,Miércoles,false
2026010100,2026-01-01T00:00:00.000Z,2026-01-01,2026,1,1,0,Jueves,false
2026010101,2026-01-01T01:00:00.000Z,2026-01-01,2026,1,1,1,Jueves,false
2026010102,2026-01-01T02:00:00.000Z,2026-01-01,2026,1,1,2,Jueves,false
2026010103,2026-01-01T03:00:00.000Z,2026-01-01,2026,1,1,3,Jueves,false
2026010104,2026-01-01T04:00:00.000Z,2026-01-01,2026,1,1,4,Jueves,false


In [0]:
df_spark.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.{table}")